# adminlineage local test notebook

This notebook is a step-by-step place to test `adminlineage` with your own files.

You only need to edit the input settings cell near the top, then run the notebook from top to bottom.

## Before you start

1. Install the package in this repo:

```bash
python -m pip install -e .[dev,io]
```

2. Put your Gemini key in `.env` if you want to run the LLM step:

```text
GEMINI_API_KEY=your_api_key_here
```

The package will search for `.env` from the configured repo root below, so the run does not depend on the kernel's working directory.

3. Replace the placeholder paths and field names in the next code cell with your real dataset details.

In [1]:
from pathlib import Path
from uuid import uuid4

import pandas as pd
import adminlineage


## Edit these values

Update only this cell first. After that, you can run the rest of the notebook in order.

In [ ]:
# Replace these placeholder file paths with your own files.
# Supported input formats are .csv, .parquet, and .pq.
#from_path = Path(r"C:\Users\Taha rice\Downloads\SAGY  Ministry of Rural Development, Government of India.xlsx")
#to_path = Path(r"C:\Users\Taha rice\Downloads\Shrug state and district.dta")

# Country and period labels.
country = "India"
year_from = 2011
year_to = 2025

# Name columns to match.
map_col_from = "district_name"
map_col_to = "District"

# Exact-match columns are optional.
# Leave this as [] if you want global matching.
#exact_match = ["replace_with_exact_match_column_1", "replace_with_exact_match_column_2"]

# ID columns are optional. Use None if you do not have stable IDs.
#id_col_from = "replace_with_from_id_column"
#id_col_to = "replace_with_to_id_column"

# Extra context columns are optional.
extra_context_cols = []

# Start with relationship='auto' unless you want to force one mode.
# Allowed values: 'auto', 'father_to_father', 'father_to_child', 'child_to_father', 'child_to_child'
relationship = "auto"

# Keep reason=False at first to save tokens.
reason = False

# LLM settings.
# string_exact_match_prune options: "none", "from", "to"
model = "gemini-2.5-pro"
string_exact_match_prune = "to"
gemini_api_key_env = "GEMINI_API_KEY"
batch_size = 25
max_candidates = 15
seed = 42
temperature = 0.75
enable_google_search = True

# adminlineage notebook runs are usually editable installs, so this points back to the repo.
package_root = Path(adminlineage.__file__).resolve().parents[2]
env_search_dir = package_root if (package_root / ".env").exists() else Path.cwd()
output_dir = Path.cwd() / "outputs" / f"adminlineage_notebook_{uuid4().hex[:8]}"
replay_enabled = True
replay_store_dir = package_root / ".adminlineage_replay"

print(f"Using .env search root: {env_search_dir}")
print(f"Run artifacts will be written under: {output_dir}")
print(f"Replay store: {replay_store_dir}")


## Load the datasets

In [3]:
from_path = Path(r"C:\Users\Taha rice\Downloads\Shrug state and district.dta")
to_path = Path(r"C:\Users\Taha rice\Downloads\district_25.csv")


In [4]:
# Run this after you update the file paths above.
df_from = pd.read_stata(from_path)
df_to = pd.read_csv(to_path)

print(f"Loaded df_from with shape: {df_from.shape}")
print(f"Loaded df_to with shape: {df_to.shape}")


Loaded df_from with shape: (631, 2)
Loaded df_to with shape: (3361, 1)


## Inspect the source data

In [5]:
df_from.head()


,state_name,district_name
0,jammu kashmir,kupwara
1,jammu kashmir,badgam
2,jammu kashmir,leh ladakh
3,jammu kashmir,kargil
4,jammu kashmir,punch


In [6]:
df_to.head()


,District
0,North And Middle Andaman
1,South Andamans
2,South Andamans
3,North And Middle Andaman
4,North And Middle Andaman


In [7]:
print("df_from columns:")
print(df_from.columns.tolist())

print("\ndf_to columns:")
print(df_to.columns.tolist())


df_from columns:
['state_name', 'district_name']

df_to columns:
['District']


## Validate the inputs

This checks whether the key columns you configured are present and whether the exact-match setup looks usable.

In [8]:
validation = adminlineage.validate_inputs(
    df_from,
    df_to,
    country=country,
    map_col_from=map_col_from,
    map_col_to=map_col_to,
)

validation


{'valid': True,
 'errors': [],
 'warnings': ['No exact_match columns supplied; candidate generation will run globally and may increase false matches.',
  "Collapsed 6 duplicate df_from rows using match key [district_name]; 625 unique rows remain. Duplicate values were ignored and only the first row per key was kept. If expected, ignore this warning; otherwise inspect or fix the data. Sample duplicate keys: (district_name='hamirpur'); (district_name='bilaspur'); (district_name='pratapgarh').",
  "Collapsed 2750 duplicate df_to rows using match key [District]; 611 unique rows remain. Duplicate values were ignored and only the first row per key was kept. If expected, ignore this warning; otherwise inspect or fix the data. Sample duplicate keys: (District='North And Middle Andaman'); (District='South Andamans'); (District='Visakhapatanam')."],
 'map_col_to': 'District',
 'from_rows_input': 631,
 'from_rows_effective': 625,
 'to_rows_input': 3361,
 'to_rows_effective': 611,
 'collapse_repor

## Preview the candidate plan

This does not call the LLM. It gives you a quick look at grouping and candidate counts before you run the full job.

In [9]:
preview = adminlineage.preview_plan(
    df_from,
    df_to,
    country=country,
    year_from=year_from,
    year_to=year_to,
    map_col_from=map_col_from,
    map_col_to=map_col_to,
    extra_context_cols=extra_context_cols,
    string_exact_match_prune=string_exact_match_prune,
    max_candidates=max_candidates,
)

preview


{'valid': True,
 'country': 'India',
 'year_from': 2011,
 'year_to': 2025,
 'from_rows_input': 631,
 'from_rows_effective': 625,
 'to_rows_input': 3361,
 'to_rows_effective': 611,
 'from_rows': 625,
 'to_rows': 611,
 'exact_match': [],
 'groups': {'__all__': 625},
 'max_candidates': 15,
 'avg_candidates': 15.0,
 'warnings': ['No exact_match columns supplied; candidate generation will run globally and may increase false matches.',
  "Collapsed 6 duplicate df_from rows using match key [district_name]; 625 unique rows remain. Duplicate values were ignored and only the first row per key was kept. If expected, ignore this warning; otherwise inspect or fix the data. Sample duplicate keys: (district_name='hamirpur'); (district_name='bilaspur'); (district_name='pratapgarh').",
  "Collapsed 2750 duplicate df_to rows using match key [District]; 611 unique rows remain. Duplicate values were ignored and only the first row per key was kept. If expected, ignore this warning; otherwise inspect or fix

## Run the package

Run this cell only after the validation and preview look reasonable. This is the step that makes the LLM call and writes output files.

In [10]:
crosswalk_df, metadata = adminlineage.build_evolution_key(
    df_from,
    df_to,
    country=country,
    year_from=year_from,
    year_to=year_to,
    map_col_from=map_col_from,
    map_col_to=map_col_to,
    extra_context_cols=extra_context_cols,
    relationship=relationship,
    string_exact_match_prune=string_exact_match_prune,
    reason=reason,
    model=model,
    gemini_api_key_env=gemini_api_key_env,
    batch_size=batch_size,
    max_candidates=max_candidates,
    output_dir=output_dir,
    seed=seed,
    temperature=temperature,
    enable_google_search=enable_google_search,
    env_search_dir=env_search_dir,
    replay_enabled=replay_enabled,
    replay_store_dir=replay_store_dir,
)


2026-04-06T12:20:30 | INFO | run_id=d111f6fc538b083b stage=start country=India years=2011->2025


2026-04-06T12:20:34 | INFO | run_id=d111f6fc538b083b stage=resume archived_stale_raw_links=links_raw.archive-069f06120f29e1d4.jsonl


2026-04-06T12:20:34 | INFO | run_id=d111f6fc538b083b stage=resume completed=0 pending=625


2026-04-06T12:20:34 | INFO | run_id=d111f6fc538b083b stage=adjudication batch=1 size=25


2026-04-06T12:20:39 | INFO | run_id=d111f6fc538b083b stage=adjudication batch=2 size=25


...

2026-04-06T16:22:17 | INFO | run_id=d111f6fc538b083b stage=adjudication batch=24 size=25


2026-04-06T16:22:20 | ERROR | run_id=d111f6fc538b083b stage=adjudication batch=24 failed=[Errno 11001] getaddrinfo failed
Traceback (most recent call last):
  File "C:\Users\Taha rice\miniconda3\envs\adminlineage-local-py311\Lib\site-packages\httpx\_transports\default.py", line 101, in map_httpcore_exceptions
    yield
  File "C:\Users\Taha rice\miniconda3\envs\adminlineage-local-py311\Lib\site-packages\httpx\_transports\default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Taha rice\miniconda3\envs\adminlineage-local-py311\Lib\site-packages\httpcore\_sync\connection_pool.py", line 256, in handle_request
    raise exc from None
  File "C:\Users\Taha rice\miniconda3\envs\adminlineage-local-py311\Lib\site-packages\httpcore\_sync\connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Taha rice\miniconda3\envs

2026-04-06T16:22:21 | INFO | run_id=d111f6fc538b083b stage=adjudication batch=25 size=25


2026-04-06T16:22:24 | ERROR | run_id=d111f6fc538b083b stage=adjudication batch=25 failed=[Errno 11001] getaddrinfo failed
Traceback (most recent call last):
  File "C:\Users\Taha rice\miniconda3\envs\adminlineage-local-py311\Lib\site-packages\httpx\_transports\default.py", line 101, in map_httpcore_exceptions
    yield
  File "C:\Users\Taha rice\miniconda3\envs\adminlineage-local-py311\Lib\site-packages\httpx\_transports\default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Taha rice\miniconda3\envs\adminlineage-local-py311\Lib\site-packages\httpcore\_sync\connection_pool.py", line 256, in handle_request
    raise exc from None
  File "C:\Users\Taha rice\miniconda3\envs\adminlineage-local-py311\Lib\site-packages\httpcore\_sync\connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Taha rice\miniconda3\envs

2026-04-06T16:22:24 | INFO | run_id=d111f6fc538b083b stage=finish rows=631


## Inspect the output

In [11]:
crosswalk_df.head()


,from_name,to_name,from_canonical_name,to_canonical_name,from_id,to_id,score,link_type,relationship,evidence,reason,country,year_from,year_to,run_id,from_key,to_key,constraints_passed,review_flags,review_reason
0,kupwara,Kupwara,kupwara,kupwara,from_0,to_186,1.0,rename,father_to_father,Canonical name 'kupwara' matches exactly with ...,,India,2011,2025,d111f6fc538b083b,from_0,to_186,"{'candidate_membership': True, 'exact_match': ...",[],
1,badgam,NaN,badgam,NaN,from_1,NaN,0.0,no_match,unknown,No suitable candidate found with a high matchi...,,India,2011,2025,d111f6fc538b083b,from_1,NaN,"{'candidate_membership': True, 'exact_match': ...","[low_score, type_needs_review]","low_score,type_needs_review"
2,leh ladakh,Leh Ladakh,leh ladakh,leh ladakh,from_2,to_260,1.0,rename,father_to_father,Canonical name 'leh ladakh' matches exactly wi...,,India,2011,2025,d111f6fc538b083b,from_2,to_260,"{'candidate_membership': True, 'exact_match': ...",[],
3,kargil,NaN,kargil,NaN,from_3,NaN,0.0,no_match,unknown,No suitable candidate found with a high matchi...,,India,2011,2025,d111f6fc538b083b,from_3,NaN,"{'candidate_membership': True, 'exact_match': ...","[low_score, type_needs_review]","low_score,type_needs_review"
4,punch,NaN,punch,NaN,from_4,NaN,0.0,no_match,unknown,No suitable candidate found with a high matchi...,,India,2011,2025,d111f6fc538b083b,from_4,NaN,"{'candidate_membership': True, 'exact_match': ...","[low_score, type_needs_review]","low_score,type_needs_review"


In [12]:
print(crosswalk_df.columns.tolist())


['from_name', 'to_name', 'from_canonical_name', 'to_canonical_name', 'from_id', 'to_id', 'score', 'link_type', 'relationship', 'evidence', 'reason', 'country', 'year_from', 'year_to', 'run_id', 'from_key', 'to_key', 'constraints_passed', 'review_flags', 'review_reason']


In [13]:
replay_summary = {
    "execution_mode": metadata.get("execution_mode"),
    "replay_hit": metadata.get("replay_hit"),
    "replay_key": metadata.get("replay_key"),
    "replayed_from_run_id": metadata.get("replayed_from_run_id"),
}
print(replay_summary)
metadata


{'run_id': 'd111f6fc538b083b',
 'schema_version': '2.0.0',
 'output_schema_version': '2.0.0',
 'request': {'country': 'India',
  'year_from': 2011,
  'year_to': 2025,
  'exact_match': [],
  'map_col_from': 'district_name',
  'map_col_to': 'District',
  'relationship': 'auto',
  'reason': False,
  'model': 'gemini-2.5-flash-lite',
  'batch_size': 25,
  'max_candidates': 15,
  'seed': 42,
  'temperature': 0.75,
  'enable_google_search': True,
  'schema_version': '2.0.0'},
 'counts': {'rows': 631,
  'from_units': 625,
  'to_units': 533,
  'avg_score': 0.804844,
  'from_rows_input': 631,
  'from_rows_effective': 625,
  'to_rows_input': 3361,
  'to_rows_effective': 611},
 'coverage_by_group': {'__all__': {'from_units': 625,
   'matched_from_units': 529,
   'unmatched_from_units': 96}},
 'warnings': ['No exact_match columns supplied; candidate generation will run globally and may increase false matches.',
  "Collapsed 6 duplicate df_from rows using match key [district_name]; 625 unique rows 

In [14]:
artifacts = metadata.get("artifacts", {})
artifacts


{'run_dir': 'outputs\\india_2011_2025_district-name',
 'links_raw_jsonl': 'outputs\\india_2011_2025_district-name\\links_raw.jsonl',
 'review_queue_csv': 'outputs\\india_2011_2025_district-name\\review_queue.csv',
 'run_metadata_json': 'outputs\\india_2011_2025_district-name\\run_metadata.json',
 'evolution_key_csv': 'outputs\\india_2011_2025_district-name\\evolution_key.csv',
 'evolution_key_parquet': 'outputs\\india_2011_2025_district-name\\evolution_key.parquet'}

## Optional: read back the saved crosswalk file

This is useful if you want to confirm what was written to disk.

In [15]:
csv_path = artifacts.get("evolution_key_csv")
if csv_path:
    saved_crosswalk = pd.read_csv(csv_path)
    saved_crosswalk.head()
else:
    print("No CSV artifact was recorded for this run.")
